# ValueLens - refresh the semantic model

Refreshes the ValueLens semantic model after the pipeline has loaded new data. The
ValueLens installer deploys this notebook and adds it as the pipeline's last step.

- It runs as whoever owns the pipeline, so they need Contributor (or higher) on the workspace.
- The audit table refreshes incrementally: only the last few days are reloaded.
- After a backfill (`WRITE_MODE = 'overwrite'`) it also reloads every partition of the audit
  table, because the processor rebuilt all of it.

It writes nothing to the Lakehouse.


In [ ]:
WORKSPACE_ID = ''        # Fabric workspace that holds the semantic model
SEMANTIC_MODEL_ID = ''   # The ValueLens semantic model
WRITE_MODE = 'merge'     # 'overwrite' after a backfill reloads the whole audit table
TIMEOUT_MINUTES = 120    # Give up waiting for a refresh after this long


In [ ]:
import time

import notebookutils
import requests

AUDIT_TABLE = 'Chat + Agent Interactions (Audit Logs)'
BUSY_RETRY_MINUTES = 30
FINAL = {'Completed', 'Failed', 'Cancelled', 'Disabled', 'TimedOut'}

if not WORKSPACE_ID or not SEMANTIC_MODEL_ID:
    raise ValueError('Set WORKSPACE_ID and SEMANTIC_MODEL_ID in the cell above.')

REFRESHES = f'https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets/{SEMANTIC_MODEL_ID}/refreshes'


def _headers():
    # A fresh token for each call: a long refresh can outlive one.
    return {'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi')}


def _history(top=10):
    r = requests.get(REFRESHES, headers=_headers(), params={'$top': top}, timeout=60)
    r.raise_for_status()
    return r.json().get('value', [])


def _start(body):
    """Starts a refresh and returns its ID. Waits while another refresh is running."""
    deadline = time.time() + BUSY_RETRY_MINUTES * 60
    while True:
        r = requests.post(REFRESHES, headers=_headers(), json=body, timeout=60)
        if r.status_code == 202:
            location = r.headers.get('Location', '')
            return location.rstrip('/').split('/')[-1] or r.headers.get('RequestId')
        busy = r.status_code in (409, 429) or 'in progress' in r.text.lower()
        if busy and time.time() < deadline:
            print(f'Another refresh is running ({r.status_code}). Trying again in a minute.')
            time.sleep(60)
            continue
        raise RuntimeError(f'Power BI would not start the refresh ({r.status_code}): {r.text[:1000]}')


def _wait(refresh_id):
    deadline = time.time() + TIMEOUT_MINUTES * 60
    started = time.time()
    while True:
        r = requests.get(f'{REFRESHES}/{refresh_id}', headers=_headers(), timeout=60)
        if r.status_code == 200:
            d = r.json()
            status = d.get('extendedStatus') or d.get('status')
            if status in FINAL:
                if status != 'Completed':
                    messages = '; '.join(str(m.get('message', m)) for m in d.get('messages', []))
                    raise RuntimeError(f'The refresh ended as {status}. {messages}'.strip())
                print(f'  done in {round((time.time() - started) / 60, 1)} min')
                return d
        elif r.status_code not in (404, 429, 500, 502, 503, 504):
            r.raise_for_status()
        if time.time() > deadline:
            raise TimeoutError(f'The refresh was still running after {TIMEOUT_MINUTES} minutes.')
        time.sleep(30)


first_refresh = not any(h.get('status') == 'Completed' for h in _history())
print('Refreshing the semantic model' + (' for the first time' if first_refresh else '') + '...')
_wait(_start({'type': 'full', 'commitMode': 'transactional', 'applyRefreshPolicy': True, 'retryCount': 1}))

if WRITE_MODE == 'overwrite' and not first_refresh:
    # An incremental refresh only reloads recent days. The curated table was rebuilt, so reload all of it.
    print(f'Reloading every partition of "{AUDIT_TABLE}" after the backfill...')
    _wait(_start({
        'type': 'full',
        'commitMode': 'transactional',
        'applyRefreshPolicy': False,
        'retryCount': 1,
        'objects': [{'table': AUDIT_TABLE}],
    }))

print('The semantic model is up to date.')
